# Mixed Emotion: Llama 2 matched CoT

Verified final DistilBERT predictions are embedded; no input upload is required.
Mount the same Google Drive account. Evaluate 86 routed cases and report all 300 examples.
Phase 1 accuracy: 251/300; temperature: 1.4801950079829693; threshold: 0.70.
Generation settings and prompts are inherited from the corresponding final Reddit notebook.
Outputs are saved under unified_final/mixed_emotion/phase2 and support row-level resume.


In [ ]:
# Run once in a fresh Colab GPU runtime. Restart the runtime after installation.
%pip install -q -U pandas tqdm scikit-learn sentencepiece protobuf accelerate transformers "bitsandbytes>=0.46.1"

import importlib.metadata as importlib_metadata
print("bitsandbytes:", importlib_metadata.version("bitsandbytes"))
print("transformers:", importlib_metadata.version("transformers"))
print("accelerate:", importlib_metadata.version("accelerate"))
print("\nRestart the runtime once, then run from the imports cell.")


In [ ]:
import gc
import hashlib
import json
import os
import re
import time
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from IPython.display import display
from sklearn.metrics import accuracy_score, f1_score
from tqdm.auto import tqdm
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

LABELS = ["Depression", "Neutral", "Happy"]
LABEL_SET = set(LABELS)


## Configuration and final Phase 1 input contract


In [ ]:
# Both Phase 2 notebooks must point to this same final Phase 1 export.
PHASE1_INPUT_PATH = Path(
    "/content/drive/MyDrive/confidence_guided_llm_reasoning/outputs_final/"
    "unified_final/mixed_emotion/phase1/final_mixed_phase1_reasoning_input.csv"
)
DRIVE_OUTPUT_ROOT = Path(
    "/content/drive/MyDrive/confidence_guided_llm_reasoning/outputs_final/"
    "unified_final/mixed_emotion/phase2"
)

# Frozen final Phase 1 contract. Any accidental input change fails before inference.
EXPECTED_TOTAL_ROWS = 300
EXPECTED_ROUTED_ROWS = 86

MAX_ROWS = None  # None for the final run; use a small integer only for a smoke test.
MAX_REASONING_CHARACTERS = 6000
REASONING_HEAD_CHARACTERS = 3500
REASONING_TAIL_CHARACTERS = 2500
LOAD_IN_4BIT = True
RESUME_FROM_EXISTING = True
BASE_SEED = 42

# Canonical required columns in PHASE1_INPUT_PATH:
# example_id, target_label, phase1_label, phase1_confidence,
# phase1_routed, phase2_original_text
# title + selftext may replace phase2_original_text; aliases below are normalized.


In [ ]:
try:
    from google.colab import drive, userdata
    drive.mount("/content/drive")
except Exception as exc:
    raise RuntimeError("Google Drive must be mounted before the final run.") from exc

# Optional for the public NousResearch checkpoints used below. A token can
# still be supplied through Colab Secrets to reduce Hugging Face rate limits.
try:
    hf_token = userdata.get("HF_TOKEN")
except Exception:
    hf_token = None
    print("HF_TOKEN is not set; continuing with public Hugging Face access.")
if hf_token:
    os.environ["HF_TOKEN"] = hf_token
    os.environ["HUGGINGFACE_HUB_TOKEN"] = hf_token

DRIVE_OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

import base64
import zlib

embedded_payload = zlib.decompress(base64.b64decode('eJztfduS6zay5a8o/Fx28H45Lx3ubs/0PnGmj2e2+/TDxISDEqESXRSpJqktyxPz75OZAChKBUlVFCBSlydvV5UkCkhkrpWZC/m//+937PdkucrZr1n63b9NvvsfP/2X8+tff/r5V8uyv3uZfNck1Strfs2TKcvx939lq4rVdVYW+NvVIqmZvfvt35LVatv5xaws5lnKihmD31o/hLEfe16w+31VrhuGnztP8prJnzu/llX2mhVJ/mvDfm/oY9dVVrxOkkk9W5RlPkmT7cvky6Qom2zGUvj5W1akkyU8WfLKJvOqXMLP5lXGivRlksCvyiqFN6y2kyap3+pJ3WR5PikYS+HVTTmZwouyIqsXLP1h8tM3Vkw2WbOYNIuyZrvXltVkVdZZk31jk2W5ZEVTv8DfsIpNNkkNn7hgybftpE7SAp5kUrFmXRX43FmBfzaZJrO3V/jOBXzIl8kbWzXwNxVLUvwb/IM6WbL2W+ATwPpM3opyg3+wKDf4qElRb1j1MmH1is2yJM+38PTzEp5hCm/1Nk/q5ofJn7f0fvT18R/tY6+Sqqknc5Y38OcZm/PVwT/BZ89Y1T49fiO+TPjbOWM5PsSXySyp4IUpX6Al++G7//cyOWFFzuetSPnbA1OKIse2LauPLf3YwF5XdQNffbIpqzewJb4gy6SAdU+mOSwlmyVr2Prd5sInl+sqecVFgMVIcS+5odVgCWXBYGkq2hNY/q+4bi9yj5sK/7acg9U2DCwGNmtZ84Wv2BIMl1Xif+FzcLHxiTZgXZOy2LevepFUq0lalSs0qeUWrLAE24ZFgc9aJLCRGRgQGEmzyIo3NLKG9gp+koKxpPA+C4ZvCL+hrzxL8uULbvkLGncFe8y+sXTPuDaLLMfPB/uBrw6LXMP7/DD55yJpwD6SLbx9XhavrG7oIeFI0nd4xSfjh+RlMl03wr65bTX4Yn488PTyhUgy/J91AatY5vAU5yzLNWNZoRPaodPHsH4RWzdlr4lwIMkkXcMJYr8vwJwa+Fz+1ZcJbAWsdLXFjXqd1IwtYf8qsASx3l9oMVOW0Kmj94ItKbinwE3mxw8sN1lm+faHyS+tmWySCoxyvVqVVcMtdMVKWMFJQq4Hziw3to5xzMvZugY7mmRisz7jLndumL5b68fg+4EJw6nJU3If8pnQDeHn1w3uxXydT5KGbHjP7JJ5A4uBJo9fA/YIVqtOcAnpu+LnJ7CwyxJ/BP/asOx10UxyBv5xMoP/VPAmTQmLkXaXHz8Yly6tkg28qkroNMCDF9z65+eszjNjdbHr+l5ghZfERh7v6kUG3qGu12w/Qi7KFcPF7i4lbGlSyUiBHkp4H2kgH3ZfFwTNAo78ul7TprPlqtm2sYb8Gj5RVYIRo0sGq/rXOmPNgWerZ1U2ZTvXKQLnMnmj4At+igy/znBLxW5n5K4Uni4vS3pZCauEBwxWkNXc5roxIGVNkuX0FSDs5vzYfMF1ZGn7JPKb5OXmZZJm4C7RG+LKrQuKM/D44NPf+LPjt8/QP56zQd9QTHUd34u9/ia4LL+RfwKQApa1mcyyZg+m5ckazigsa9rCOXB6DF0SruD1cFrNCvz7Oe4JLEIBQRtPAzlL3AnyyiJMSdtc4jaBAQL+YUqnt/NFDcKBGVjZK/w+ZdJvwyds8I8RKCzZcsrwqwso8s7zSVgIplrmLbTFD+YetcDDAGcelplCLX/kl8mWNRLxNWL50ClI5y8tUvpABBhZxQRKbr8CPAbY7Dk7DPQzBHCwkWf3N8GKzfDg0sqJCJJm83k2WwPa2TD2tu8U6yV8VwhMs1kLSziQmjK0LQF6wONd0SNyAyzxqMgnwh/DAtDOTdkCuQ6hB7QzBuuCZjHnwX1bs3wOMH9LS9A08O4EO0rxXRGorWGD4JMrxl3hNHvlwVrlDlfrpuH0hI4ERM3tmOhFaMgVhkFgh7Z7Eb2Aj4H1EAaEBxm/xQeoBi7IrFzOEfiAkXGHRzARjIUDxtamW67xccf5QaohXSTihhzWawfcD+A9wNQsJdwP8ALshg5gS3A3JR4HcjmLbLbYOSDyYvwxp/LbNyVYZYXB8p0/xA9dMka2SEF2DBQkMmN9vuNFsa+iIE21/jQDEf59BU4JEAjutVyD14pnFhAsCfAHhLjanuAg8GUbHroxwP9WTrvMI+GHfgfg5pNkBpa8ysHw+DacIx6f8rCqN8IvUcBygN2uK2DJ8I2TFtpuFiVyWQoD7zxdxcBBbbiltnEgBRc7LOGITXm40HKs+IJkXMPhNeIhub6w5jWDNU73oyxhdnES8U/5F78m4OtY5JSR/eYYO+kBwFttFixfsm5WTmbuKJciCK0Mytxc0pLecO/xFX5r573JgyWYCoQ/HRWhsC1DqZTYCmI36m9iv8Eac28DXmS5LoBPcKSyb1/wnZIaVgIWL3/jnh02dEXEgrKisBhXBnBrWEvcDHn06bEx+1bIbMyGAl61FVaWFbN8nWKM2/NrBTc8RmaK+K0L4NFvZ69FWbX5Y4R4YHRg2yosN1uwGZ1ZoLWzJAfTSqpRMQpbf80hcgI3tuP+NtghAZNsiQkstDSIfvV9VCCAe9fim4rccIudEM0LoF6VhLvQAeLTcnK7Sxe9c3t79ASC5HxMRQnbVFHCDn0r6pNA6ZCGPCmEz+OEbQboe1rxdN39lykQzgOMKBDLJ9NSgPZl9jtDB4XMVWIN9jsg2owd5o3b1BIHHVQTGwFVsA1VKyLH9YLIGV+1gqyU7wQ6nBWw4fI3srohKxVk6DU84GzBcynNhid6CvLxyFWSYstBu/hZDV9QXZxoU9OvYEUiLz0gU7CNlSb82HfcC0oTuzAk4iKdygcpT4CfyhMRfvG8wY9Ekg5+CBAhyeAUHcRQUdCHJSkEXMuTVVOuxsUaTJUh/CC0vUvsra2UyiiAjB9NCIlBuWR3W5OgA0FOrKUEq/U0z2YTPFP4drtDxJ8Sm0qyhgOLrMDSvsLRAQqrZP4RIQStGR3O6XZc/MFARcL2Xcu7gD/sEgAJ5eAoJCwhqNRYJrrbckTr/zAdNJmW9LmsU4whctHifOH9+Q8V7QDCI76Wgi7XOWOrMVEKU4UIK7Cc4DJKUcAXfk1E2ha8S/JaAYJD4P44tQh5xtDfQUyYvaH3FMREVmoPUF03oqsohliaJezHGAiGoVpEaIde7PQJxsaLEVTdFHG+YutC9gYNVo3gJVgIp0iOyEthARBewysKC4g0DDeVYjpa4heIuWBzsAuwpPSpKia7ojMpICQZaVEO3AZlm6pKRI7t9Wq+k9m6snrjh7OiXFUymWH+l0ICJ6A3XpogGxXdJBTxkqrpojryJfiG1F0iXJEqF4wmIshFykZWkHAMFSQiN/bsoJc3k1126W94hLpdTvgW2SGSu7+SxF6/E1ooROUGV3Y/vUj5XhEIBXRSAbmKrXL5OUmxpSrriIiEY0D8YNlWeEl33f2JHyj9S1+NMr3d+oPwyPyd5TutAP1jxm+CZ1AGRmWzev7GFRNouqPhCI6pskMQWEHYJwf8UFIIDGLZH3sdm6Ivcxc9l4gpKfvTdseR7TL4smiCaoBWg0PZC8ToWeFXI6AFjimVhO2EkbJF6amSOFJ7eG9/vKyKNtHGLmEbSb5JtvwE5CXE4JpCrsLViR73hiWD8gHHWO3Bi2P7gg6SR1NFbKqywd6kTXHAOeuyxVdrerhGBjvye+juhHmckRaOiSaYqkBEru37rv9UQnyo6vBOCbGCswE/AkfCjaPmrrzZiSRqrgjDXB43u49IDsfED0xIH8LAdp7Shx61Bn1K6lOqw/HwCFO1BsfzbfsyHvHomgddeusjKsTh+YShMkPg2V4QqSLuU/KwX2QwJ7Y+JjkckleYqzNYketfkI27T/WDKVG1UkU4IibhGio4BHEYW5Ey63vGrz26AsKwvLp5LyccEb1wDZQfbCt2vV6dJPcvhDCknT6hKBwNmXBNFSUc2/f8Cy9oekwxhFbd9FH94eBUwjVUmvBD33EiVSKlD5W4d0WETgW1Wl84IH1wTZUlbN+J/QvKEo+oiNCtrFZJDMdEKQwVJ2LbCwO3T5bu0dURukXVaqnhmGiE/ipFFFmRZ12SOHlQPYRBrXXrcTs6xPHwC0PFijD2AojCT2GEroLFxcrr46LF4amGsapFHDmRShw2dNVidNoIrcrr04LEIQmHoXpFbFuO5zyFEWeqFhdorQ+VhSNiEp6p65kszwYa+xRDfL46oV1ordIUjohJeCYKEn7o9Oo2uUc9hGa59Fnt4Gj4gWdsQEQYhxfygwcQRejURR8TFg4O/z1DlYbYCmPH7tOyabjUMF4RhF6J9Akx4YAkwDNVdQAOEIdPMcTnKg66BNP7osIx0QNTKgg7dOPYfs6D6KGC6K+P/oCocEy8QH+FIaDezAta5h5cB2FKVN1VGo6HPZiqLgCw84Neav2nFkKWFrQLrt+LEYenFqZGQMSRHyu7NYeuLIxND2FEVv1OezgknTCmgbC9i242vE8JhG4F9blBhiNhEb6pIoPvxJfd5vqUQOjVVR+RFo6IVfgGqg2Ra9u+f4EZ3rH8Qbd++piQcDS0wTdUdAhDP4xj6zkBYrhB1Uql4eA0wTdUgQhi3wr8SNeouHsXO5jTTjfv9YQDUgbf2HVMQeCESnN7Ch+OliFMiarPDCwcC7EwN606cnvdevjoOgjD0uqjisMxMQwDyojAC8Jew78eWxhhSHut1iKOh4GYusPJc+04sC4bXP3URWgWYaukisNzkh6lizM+0Lec0AqVQ4WHLluMThChR3it1iAOST3M3dgUeU4fv/YAAgjd0uqWyhxMNhwJpQhMjYfwYjeKntMhRjCw+riYcEQsIjAwrtqOfecpijA+n/q0gnA0NCEwVKiIbdsJel1w/VDqCO3i6EOV4eAkIDA1lxpogB/3gWuGCxPjlUZoUkirlIQD8oHAXCkicuJLShGPpoa4SCX9oWGEY6EHxgQRdnjRBK+H0kNol0gfkxeOiRCYkEM4sW31qfA/vBzCzAhqldhwPHzB2GiIyHItu09q5KmHMDGDWqVJHJ5MGJtBbYVW3Ef4arysMDY1hKER1MfFiENSC1OlBhcirnfJxTj3qYzQKqc+HFw4Eu4QmiotgHO0okts6imD0DJ4uiMeHBFxCA0oHnzHdd3nvIcBpk8faAdHww5CU7IH148d+7Keo8dUPWiXSJ9SGw7OD0JTxQY39pxgjNWGccogjEimj8gKB+QHobHSQxS7cXgBlntEFYQZXfWpmYVjoRSmBkFYgR9cIqx+CiD0CquV0sMxkQwDogfH96NembnHFj3oVlw37+WH42Edpu5oAsvzrD63bz6VDiZGVp+QIw7PPExd02THYeiNcm716AQP5sTXRxWHQ5IPU8WJ2LN95wLucc86CFNK6+ODDUfCMSJTtzdFwDEuuqT/YRURhlXWKknhiLhGZKCg4TqxZV10/fUdaSO0y6bVSsLRsIjIlBLCiQIr9i4rXjyAFEKrHrpNEnb0hYNzhMjYHU1RpDKvYS9oGq8OQo8w+riAcEBSEJmqSDi261xS6380LYRuwfT5gYRjIQrGplI7th1dQBQeShthSC19qC4cEx8wUHtwI+8Cl/e4uggjsmqV7HA85MGULMJ3I9e3LtNRP7osQp/I+qxIcXiOYaoO4VlB5DzHRVxtDvUxMeKQ5MLU7GnLD63wgkh7n2oIHUOnjw0uHAlZiA1VFQIvsK3Y7q9pfVgthCFR9b6OcES0ITZQRgg833IuGHtzx7oIU1Ooj8sHR0MRYlPaiMhzPadP1erRtRH65dJdzeHgVCA2VG4IozB01C1JTzHEtSZSd1SGAzKC2FC5IcKxwM4llOARBRCap1ErBhSOhTYYEzzE9iWdb0+9w2UK6lPawjFxBgOlBisOvV633zxlDhNTOusjgsPxMApTugfHDYN+99A9hQ9G51OfliQOzzgMFR8C2/M9V0Vxhy4+jE4DYUR5rdQdDkk7jEkfXCv0nyMgrji2unmvKxwP0bAtU5c12VHkRf4FVOMpe9CsrT4hLhwH9fj7T//41bKU5Yq/szVsW95TXm33Sh7/82DVwNXlCRUAXuH0idTxCnDJi4of4ukGs0EXQqQXiwUbzIzN+H6RaSHGA0sA46V0R7FGxYoYJL7CeEn1A/g9BM8ac2Z0EsBLvXA9lXQPmG5ryMXy58FlRz0LB98UwmaLEh5wMs2T4g08Buwhj53wPukaIyCWWLJ6sl7hsRBOivY7q0kPiA6N2Eer5AKrhW8iEEZW4DlK2uiXlozHYHBUDRqV6IRqqRucTFqKc/agLCqcsIf3vzrMgXixE/kXdfoKqQJnRyKrIUVtyO03LCUjkH4fbMK2/s2yWrqEwDOHNfuWNUy+AP7CtXDvc1xSWTvKluijurst4iwl8AsmMik8dsnEQkH/U7/tZAZUGoBf4imHwApsRGwxorV1wVF1zcEZPTyX+qBtUHNGBpS6mS3ObZYyIX/JZvl+5Nm+8tK/M8j4S7toiejJR8//DbwkGiotGs9b79wsnG5KmvNzCAeHYcSv6S12R5Xb+S59JBaDTtABjCDOvynFS3Afqe7MN3IpwwujfvxqzdNSgHeYCOyYqBB9XevpMhPHGZ9+d4DF08AhpZWrlnJjcR3wx/zQJmg24KAAcsgocm4vlZnvi/bSsm0/8Hrs5Y+i6ktZN1pR8Ex1RqgHDwhv+xMlETiI+Y79keH/gR1xPOiiS9pUGbIaWcClnvX2phFMyR0ev1WezOSmUUcKTzA1iLsk15bHkz8Ednry/aAfytYC+PcW85+FqGpwke2KNp6z0boT7fGNO341ObtlyqzxJVvmxW7gB8oRwWePHyaXGAfZSOSTNK0InvAG2PYQ1oCP2iOYtIGMHz5EVOVKcC9YeXSqtBm0vxUDeiGCawIbu+BJiRpOeUqNLautZLn4cS8Qxyq5Tfz9hQVQ3otvZNfTIo7kgRHti+dsqgp9bwUhsxMipVaa+AZ+MdJ/tB3o4D1KAJYIUhPwIjPCjmlWsQ8FQGUutj8g8i0rDm27R6PjFwGGaHEx8Y+ZIFwuDFgIbyvu7cBpLWtekJanK10zgtu0bR3Uw+gfsJ3voQ/tC//pPvKpSzqfROrxg6dbeusO6kErwYeQ+427lyc8sYRfQFB7SkRklIsoq9ekyP5o26/ltzq3N8r8ZP+9gbPm2Mr7WM4HOxlZZPczui5yULswIzApWbQi/hEGnZUpJzIQ8uu2ukgZaYChW0WAS75BMKEPEuxkH67MYOnWCCFXfIdEQpsfut2zVZScme+D0o7z6wQ16Sgxrs+aNQW01o2e2zFlAu+C0xT5rm2FPfrwvpCwjs4Sclsg7Hm5ZR3H+I4GThG6S7x2BJzQc++wyTtgQQessztinV66p46/NXLBunOokpYdEyGv3zoN5tIt0qfv40oEJlgqqziqXCVAABA+n9snZQKs/z7Frut7QR/Q38WRbc8tMSKAEAn6LgxD1HB/GM0O3VubRScsLTJDvLlNBCJqhcsPCaBwgy9tZANrZ6wAOC+AIsRWsXfk9oQWb/e05Mr2GJ3YO0yFyNwubOKJ9Dftiq3MFl0CMEI3ioKgzw1Ue9HooIeVuzgRkRYi9KBTE9RK2Cs2mwgLzsuZ6AVrcQfSW1yFhh0emnm+xSvEOkdm5wMFzDng4NKZikxfF3a8B4JZ/SnUZ382Y/IBoO7EXqQ8LR8nXQXmPrAOSXBuhxMOAg9GlhbHpRCS23ImOqB059pgs0QOhQBgvYtE/Iygu9pD9JwEH0C6EotqbB/UUToXE/gbSvK9dkIN58MEEVDoBq9NPhBk7M/mLM4FmcAOglDVb/sJWIDEZMaqLQdNolJUN5i8XmWzt/VqhwtaJ9YCKRFnKLtAQQvYZApxhZb9ALTRC7jL2kcC/1qjGqTZHuAAep51IfsRd9S2pF44eipymOqUBRYPvi+RHjanynd8Y7TnJ4LId/rRo473yrNphallcBys+rZLLKphgNitDcTicsML7QuqWUi2UVBnF+K/g53pvr9qg+ApYBs6OUQeVzjwbqN+I3KO8O9sLuo1BykHmanfywrSJ53bIO1Jh9CJo9DvNeSwRWjdEg757PenhOcyue0j8ylwsersFV8Om8OJaYd+5lTH2E/rgUGUiJTFJ82z/DCVy8+i4qg0LFnKhAN4weYIN239K8/yntuMz6YTzrEb27JcZdPFZ6JKWeDaSS6zLIko7EI9RJI/wPabnLXJ1nqbwyPCyvIfUxqozrGOiw32Mr4cgi764z3qyYNJJ4NHJFhiO9G2Ct8Zdm/dhhF1hkem4s7tgG7uH0e2HV1I/YtOlmYfBiOC7GyFxMWEvuTaAZ7CpeA/6xRHKux8qoXTP/BbolC3d1oIKE8WGUYwEVE4OuafTI/S5f0Ciatyo+KvZV3j3J5o5vx+6Hu2qxRAfya6C8X4jjvvmDW+HfJvFfSC5Vvxsh9Vy6dT3GdeAHpH9LH3Z/eC/f2AL1/Uc+SG2fLQa/EHoBMk8Nn+BYD78bw9MN/O+qfPcvnzpSE/jN1eTYodNi/rM3NgXpT6PIzhWBDAvkUKFHW2BOZQCfy6nx6TWeNOcqxbG8I3ESy9uxXgElugKo/OHuDiz0XZUanvrmYLzEnyR+4Ejw7+xU+TebIPgeDPMvjzWMvzPMu6FGwtwUEviPml8GQ8KyKcFuciM5atmi5bpJ4P6nQqE6zzNey1rLbvsBV/nQpXHaQn+Qng76KM5sIhZXX73KrsV7rE7hgqRJ89Ks5nifs5p+VZju96vdJeu3SKqMe1qKRzYvbA1XvcyoUuoozDuQymPeENJ/UqmR0CX/Gi5BVWrlOEy5qdl6SK7GEcEXkClnLvBSAqE30X8nvRsXiX8dov4Ygs5bkt0k7jI99zQqtPs9eXtp68WlAzA2+nXi6RpbxzaHyLZKqrTSDXq/INdkfozkgbuV21C5tigRqbUBgyb+wFFOg4owqrBLO06Qf+q5sqllFltqjKoszLV1p1ASLeh3pparx9YZdcPrM1+nsQ/NgCSt8vHSnNl7ejyHDDiUlrz6IrAd6W4LIq9AsdBEfFxPB3oadNheyDYhGRDiIOZzqn+0/4p8iUy2GR9D1W5mDgM3kwRzu5pxmWPeGAXB+RcIIP4nr7JeLRDDtJ25ADXIR6v+jeUe4Y8YpVcbFV0iKCdJfXR0MvVB0Hu1zY/hbtQ4mTlZj2Ud+hARWVFDDjIx0FzmfJ/dnbbK0gdpT9BJ9KHksNUwudTwYgmcOntBVlTdZZnvLuTkTB74JPmxHeA2myHCZqK52+nU6U6WZQeT7nHQzodOy0GudPeDbtHQNx4Ee2c1m+pZOogPBKF5+cSh8TMsXYsiOUHLIh0hZAK2OHteUWgSlOiwAivHa2rt57NKpNQqiqd0CAtyV0QBwVMN+15eyhhA+mAZzPpgHO71IcREHcc5cWCQ4WmBC5p3Y/IUraIkLe1Fiwn+bl7E3cO4UZESYbXnhFekGNshVDbripdyoiCEkltuHihRu8lbbEvkPcZNptDpXrTmd12ZZMGHVBI2+lV1a851Bm9vmlMp077eEVUwbfJMMrKLClB3U3ZVrLsPgnsJZk2SqkeIORrDfTfTUF9wfd3n4kfxIIfT9N8Fwill8uke1+5Dh+NrfwkY32rSDqF8Owi6DtUe50vqOEe1ZhkBYeEzuBWVfTzTcbPmm1brotarw1mbe3Y8ZStpJ2FW1UvKHTA8c5x8ZXysDluCqVwNtfhLRJ4JUZky2qJLRKsSeEZ1l5Yr292aWSv8mK3ZP9afIjGuksQ4kWmy0KwpCArnYFH3E985TJm/zahqB1RXgprcAmz+2u/nwF4MjQ6jU2lK9/ew0OR490ff62rVZ+S/K13Nvlak2JB57lXAJ+z3c5zgMsIFECrxPME7z2RB42uft4M1i3S05csiO3nJrldxu1IClH25KCNdofJv+kPcnwvfESsoZ3JrwxttqnaLMyXy8Ljqdo/g79i15HXwZOcr6ts/pP9GVeS2w1I6NbSjXFfhKX176o+xDNQGp/u5OocDmn4DrOWYT2HAmc99DpJfAhg+h2u3NIIrtbuZPHXfrx5y8Tutwhz5YYARPKeq+L9tgD2sEdl86atpIa95IKFh63CZuulkje0xcurkFPmwm1BycU6+WKN8rLvDq+2+56zJTUgBlVqzry/z1D3LU7z6iPlp/ZrG5tsmak+kJXT3EGa1m471TqFRrY31c8AdAGC/E8VEGRzyMW6cxuu9q7J+D8+27cS9C/73/3F65imDXnU2B2Hlbe8wceHlyl7POH/0ny/GrB/Svf086it+IZ7i3gtYC1MIXN3ReN3BKfx5moiErckfMz30KZXWcMVeQ4PmvrsLteailigP+dladkzHzjtad24JgDeut3zNXwDU/233755eevkxmrGh746NRRrrFiov/hWqGd4BsiJywcY082H8oxJ/9Nw624WrPuepWCZwHxYlMiD8CSB4Fxrv58URw6Tm+vrgRxGOgRmqMCuqp5vki0jQHNemNVIVv5Bg3vtMfyVg5E7qjoYyRuX1EoxiseSEVOH/W6rkTKkF8tl7PDJMW10J6rPx0Fsb1ntkON9XBT8NYyVHF9EzcuUcKBn/ia5RT7rhvV6fxTE8kMvo1Ik+0lSGTRakWX0fy+2y3yQe0hTsUVAiU/2BhAro/wXO0NLWAFnu/3GRPwIYRHbiGZ0dGEAJql8tIoBtyYEh/w7gvZtnhTdP4qGE9/Qg0YvO9fcO7PYjx0A2/fY84LWX1dTzr7jgexqqhAeWOc/hq4zkReLgjsPjernkrLwf6leEkVnWGuQMIdWeX8Hjn06/UIQr1OJj8E6jOQvAttN1BOvroseQffJ8FvyNt2EiFJpzxPXebJDTP5K2E7E5m8wLX66VOOJ/L+/et//r0V0c3AxnO8yqIl8DfO2q+K4wxk6kKr11jYDyfq5oxP/IAA/Ubd5FQdI5EEOgA4M/XtkflrgDjPQKIucEKrX1zvm6fD3xWzrTjJdF0T7PII4rwBSn8FsOcZSOKFttfriqRTSTxso+Itu+h8eVSHY1XfF48fAN95BrJ6gR1EvQbnns7rYWdoC3jkTv978i35SsGRtx/fHIO/DrbzDOTtfDjq/Z2/OnGH083paMgKnOgKpD6uWVlj159oLs+a7Q2S92vCO89Ams4PnZ69gx9N01H/Z7acJjkJcDizh4OHEJofCDhDVTa7P3J/FRRoJJXnWXG/rqsPp/JY8S2rSt6O+w0O/w4G5GWScrWVPBb1rfL8awA+A9m90PH7alGPp/c2bMovseQngh/FXQ73hln+ECjPQBbPx9sUtCfx6hlq3hH8TPDqCwkD5P1lKIpI8tk6p299c0z/SmDPQCIvcKPQ7e/j1Zm8mp+nOvtjF+fxZk10wbg0q3IjW/XvkdtfFQsaacrzjUFBSvXRJebgBmsBwScJL+WVcKo48Km3Nd7Ldl9JgGvgQN9E214cx27/bP9ZHEjN+O1cQhwZLMBBBYRgKzo3by4BcAXQ55vI8gGV0In5VriEdFV8+uus/gZ/VaTyins+TatuMIJcKeZrI/0DoD3fRE7Pd5T3FfdHez9v4SCg8ozOAb+EHn7JpV6kPN9dFnBvXP86ONA3kPQLLTfoc+/8MRj4l69fJ69V1olplOqbV7x+367qTXP6awI930RvXuj379c4h/SEvI7686SPJBIwGy65b4L0XwXVmcjuWXZfOXLfIi/dUJHvxi7MSkr5tTcd3Q7hvwa4M9GvF/YM9scruH/9+1dxBYyo3lblKnnddWXdG8EfAveZ6NWz3Ki/FOtoLbft1cO7l39nbQMnGA6OzmpvBbgPSn8luGcg7YejEnq3cyngXoO354irtjqev25dP17RssrYjLvFP//8003y+asCPiOZvb792h8AfCjII4S3run01SKw8tkReA1v8spuj+ZfA98FJnr4rMhWXtSosXo7z0v+VfkMX8rfEdGv6SiwmhXjAAK6dfeGsV9gQoMbeb3miB0Df3/5+l80F5I2XOXy//HLf/s+opX4DzSS7+1bpfoDQL7ARKovDPv37B8t7C7ZEu8lAoRdtpineqO+bdRibW+Z3F8H3wUmeviCQHdRl6aLNfAV6s7RoSv5qXz/j1/+cosk/pqILjCRwovd2Oq91ecgXSdRx0eVgBdctfn8EcR1IwT/KojPREbPCxzTiI/uW+xqbsmF4x2nNexdIyv698Tyr4H3TOT6PLtfQeeENJe9rjH8/bHfq0/3a8FWMLw9FNu8bq9YPwTEM3F9XuCF+gW4/+unr7/gRVo7T7vCUfCIo/b02DdD6q+E60y060V4k7lGZKeo0uDtsaz6nu51pSpNjmea/z/85Whi/y3fpReYSO5FUX/R5kf69r7+z/+Y/FbiFVvUxIP3x4pRV+Tab5feXwPwhQZSfF5k+wZluuLSLfGZnEAD8sN721M5yYBmS+Ptw/Stb5nsXwHmhSbSeqHj9ptIebym+9/RRdJ4NfTlOHYTb/Cm677FT2+N6g+A70IT9+rZrtO7bHe0aJstacAkTfkBOEUwAS+Bb+U490bxr4P/QhN36vlR0LM997g4dwUHHC1Awmb81DxD0caCCPO9kflrAr3QRM7P6ivL+6BWt0CHnh/SfHlDS40D0W6R4l8F5hnI6/mRHffvzf1QWg9HtKA4C1M8wvWL6E6TH6bJNMuzJru58v01QJ0J5a3lWT01WJfMu7gbOj8E3DORzosj19XbpPfRaRi3R+OvBOpMXKZnuaHuau1np2LcLGW/KpgzkbULnP5dGVomYNwSi78GhItMNOO5zngmX9wbnb8CyIuMNOTFnk6M98mhGPfB5wdAepGR3jy3Z3Pm5RMzbojDXwfjRSYSd0EQ9r5U49JpGDdF168J6CID2bnQsrz+ukq9Ey/uiNBfBfmZaMoL4p59WPrHYdweqb8GtDPRe+c7rq1TbPG5wRg3SeaHQHJGcna2bSBn99EhGLdD5q+E5Uw04dmh1a/l4sLhF/dI3K8K90zk7+LY3B0qnxqMcS+E/hpILzZxTd5FVyjpHppxQ6z+CiAvNjLU1taqsPjIPIybYfIDoLnYRF4ucN3eZ1rHxIu7YfDXgXuxidSdE7g670rpOw/jNtn7NfFdbGIwhmdZtjGA12cyxo2y+6vAOiNTMCI/7p+6NzUF43aI/jXgnZHpF3HPxowLh1/cG8sfAgqaSOxZYf8JSVrnYtwH0b8SIDRxeZ5v2wOOzLgxZn9VAGikQc+zRjUO44ZyAFfAfLZlol0vtLxxDLy4G/5vFgX+7ceffwWeprKEv8H33qrs4PAXh1YQBn7Yq5EDCHSOu5NQZZYofYL99mWZT9Jk+wJbNKP1hcXI88mcsRz+YMGSbxgH0gIQFlrCuiroLkW+WTt7+WHyY7M70ngt14vw1hvaSrC9FFxxXZOqsyqXCLgqgBkpjxJlBW4Vz2+T1G+1eAg0P26cU2JwWb2gQAB7vmrgabrOmD5XfgB+Pdytt6IkP0eXBIFnKWoIWC8TVsNJzwSaJJOgVqV5Ujc/TH76JmXk+LZckIJ5hxqXCG+a4CEN1owWide5wYa/Ia751zpjDbwtahS3/JstSw6eCu5cCHzwTz1nOcrUYV/L8QM79EJVyqip1qcM56/riocKvEJPWMoiSfF4lsSn5NFt9xrb+lYAhMoV2slyC39Ywg6j6+YvRfPA/YBoXbz9MPkzP3PdN8E7HAHFE/aDVU7R0LjZyED8hTrNYE35Kn+BP8ubdmcAIFTkbiBcFeDAJ8krBI3WdLh8vJxjsYrBxhL9eRE9ycuM0DD+cQMWyuscYC8pa2OX+DTAQMsXPB0v3M9IK+jY14aO3SbJKIDAkuONJZjOgigJD7mV6TU0XuyB5Cu4gMeer3PALEUtIBKu1Wu9d0aX5TfGMY7whvxS25NMhFuWMiHZ17KC0LWVurAzhoXOl+8TbERNYB3PMtoGXxI4YMkyw0VMOArJZjyyUOTHVyWTdJ0jY1vAT8iJ01otE9gr2IpqS6uG3SkQcBK8Ie4HvPcTfcDO1jYJhKN6vcL8AbexFSsJDfFIiK5sQ+lHel8CAUkFe12Re8zINZK/zPFf2N9alTlXsZAzQV8ETOHjLg6PGIScbMb9Rcex8gvvuBHQR4rnxnZLtN66QaEkmg68Ck/cnjFyeC1BHyd0dZfkw8lJMx7upkhtGUa7OR0yjOJZIxwg7Y8gQLh8GEzo7MCqzPHLgAtYsU7iTXjKc3apTJX29nhOEHqxCjGdMUxlqOTBql5k4NSAqDJ1wARfU8hEBR8kJWMEuT9ckKoEC0MbXjIeLs6FTukMuvsFdpNwCM0weM6l45POsK+Xa2kUNwYZR4EjUdQHn0VHoqasimDSGXkthdeT6RVwUxWeZuKWwtLa00wpfUDmYJlo+Bws5mze7D0E0n0kFMIjwnfitvYizI9/K/ALsDjrmpbsnK0ps7K9bS2MXEdZeP28rYFbF2mZBLZ5gwWNY/CsXY00qwXEoHwoOjFcPhE0yIdIsyS+C4Y2SwCBnDS9PIHzTXFFQgBq2EePmOR9YNs7n4b3t2QEphuM6TwNkMJyyGhAiYikoWi/xPoUfjzwh7pRODYJB8EmsfVUgEHBT8EGE4yZ/GZgcWIEgsHVrOU5ewGqQ1+7A9/gneHMVxxrwttvGPwJPmi5ORtrlVng/vjfdsN+ndwKS8P05Tc+BU2Ght3B3DD2pjQ7bljo+xu+k1/IWuDz6Z2B+CHWp/CLmwpLe8bOEn5hE8SyGSZy5ZsiUpzSmAaBnuZl1cu/zXkY39Ysn4O/oaQnOhxePMBoRR//CjwQCF/Nt5zj9OyVh1yVewOeKusNVDLdJNsxsQdltvmo3f2Vydt1jhufF7txFKjKjh+jEJir51lGSrgXlEv5AJ2QTg4PKEKsojWShlJ5ZSpeW7foBc4xZojQupX8gueTl/OSo0/upAgqlpgvpESJPAmfIhj/+XkeuynF9YUCZUq72HQOHo/JaJYlWERVbhTujwyOMbJJsrTBaYYyzd3b9VnAX3sVO/aJBv6DRzMMrr+V01P0oskIlWDOJ6vRY7S5kNeKpxYQHoloAgS52ioIBixchbi4hYLwzjNMuedgCGi3evjFJ7ziSxf1zcvZms/LwZw2rBnYxRqT5/CFkxZcbhYlElkKCu9c4S6jvNn57hS8/0h4hTID39cQQ8t1vF66OUUIbjiyRjAmVxpWv2aw2ukR0NcxoykjS84xzhIcA38H/jNfsrMJOYo10ud28HTvZJxMDVJeR36sgAncStKSHnvvzRV+bOeMyaPh1d74p2MhDrYytd+bOFhu7DjKmzY+zxzwHlTu2rAsjaXbLYcvR3HcGr4FJf/h5WmVbOjPeQgWSZANRZ22Uiw7bpZnMR28OKkx6b9J8jceXMAyV0QfKN8Kf3wZXS24ie3AfRs5BW3NXouyahPE4lJQOCsqNNcWSJZYWcqxpl8NTB1svaWDwLI9r0/PoMLSOpAcOywARuFOQ0Ctb7CQAHRW7FC3Nkm4kpwMfnANMZdwFcUmrGMRV93lXd65sT0CRI5pROzA1lpbCC3HDXvd8S6ZQZ4UwnNxLjYDoD+VXTwPWG9AzA5epcAZFruiJDeYgktvhB/ijW3sMNXbpmx4nKf61tB8wNZbdvBcy1ffRPaZwsMyKfi2JwjpYedWVfkbqlgfsehAxlqDZfDp2O2cTHDuhFKxTajg0h75sxpig7rO0Ca7X8tGZrrHwAZsvVUGLwjCINBTZtjFRRHZ6NzeY6mBmixF8E95E5dI0JFnmSdZhc1dqlo9ulgB1PJk1ZSr0TADvSUFN/JDR9nl/3nDasurMiAgkcf9RFReLo9Us26tvkCmuOtlpD6e9TTPZruusPm+DYguJIrlWYG1eYUjA4dbydwgQgF6bjoW0+3QFEFvdcGLrDjWRBF2NL7TM7VEERnWeu6jstC6M0zXTKalAPYHrkh2wUrvyn+oqM8LB/daCrJd54ytxsQa9NcU/CCM7LgHiBPMoQAs9JqIhC54qeS1YmwpUh4PW1eQhVZ0b2AAszf0y4IXySLrAUjrwgQVkxDfbAm7MziP0FpXCB0HmEkf8rrHI6igKEJsxdYFaQUesrDAy61ZQ+SH3BiW/VCVR8UBPr4By/iIItBCv3RveoYXqonsihZPijvQeItyLI1Ltt4Cg+9ghUFTpq6s3vj5pasYEPVi7hUbxgTTvaUyA9WqRGMIwbuuIBhpBFoYPjY1isADY1Ffle0F5C9JRMrGU1xwPldc+FjVPghcL9bDI5L0t7VQbMn2JHy37ASgu8USw16jElo9bHBD2pO9nCBldvfDoArPVWyVyy+XFNu2LDoYa3A0FxZwyF+fu6fuT5NAGWZylkJeu6snSBhKHy2fmW5HxeZxPFYyyCkbyPM3LmTAUzEaQuBoLSMEXuD7dtCjlfyRNQoVHp8/9roqRe/kLizKSNZpbaOjwuDh0Q7ViKvOaQ7XLsKi5yZ0OCz+d7TWEULPiQI36KOd3W8seioY9osJ7w2TF1GxptVW6uHpSVyXb5Itp895CcG/prCrcISiE71hyTiQv6NZsuBEXhTr6Qa5a83CpkKpP91Kts8q6WYB/iU7txq0jpAiHWL2XFlb6OgAR0II9NYU/DCINZnXA8kUVhXebYs3HvDrQ+qOH+UKhj33yZXv50WAw3IAvZWDIHI8R1PK4kFkCfpEzKd0fePhCiaKB5Hru6ph409BQr/CgS7B8xHp38C8QWvdIIosK4ieegRdZQNzgudTusDB+YPWyoHvxY6vHOn2eXx3B8oEU8JmpfJvHHzB1apOCHzXciPlbPenOuEDpYMOQ9CkcW7ei/wGpRGu3lKCHwV+r4vN7lijYEi+fEKzNxrO4OqtL0SRbV9wBdJTpdAhrrpUy0fVfsNyBVdrjSEC9htfrl1+ahU65QWd+uXj4r+h+YGrtb4Q2J5lRb4egnD3YgXdwmaVInAkrOFzVYaPtR2FsR9GeqjDY8gXdAub1RrAYemC5tuQQjsIdfGFRxAsGNQ7tz68I/cbD5EwcBtS5Hq+pxyo+pQuaKhAXCyBPi4XHJha6JUv2KF/ObN4qhdkGUKrBPq8cnBwfqG3/mBHsRsoJ9R8HvTdjXJBh+j5UBM4Dt7g6a022JYTarqv9zF0CtoFzyqx36CEwdMsVXDcUJn+eCipgmbZ8lmJ3mhogKe3nuDYoeU/Ryr0qo/qkCYf0/INi++9z5UOPtbo5oahp5JYPWcsXFBC0KtUPqP2Gxrpe5orCa4d25oqCXetVNCqW97XAY6EBOiVKDhh7Ad67OpBJAr9BcofkPoNi/41lwscy9Z0u9EjaRRMiZq7Wr7xUITPVQrOhcnAcsPI61EFfUoUjrg+7Xrn99K/gfmDZpmCY0XK+8WfMoU+9QEjqmalNHBwwqC1NBCEwBh8PROy7kCaoFu/fHa03+A0wdc7N8EFrxZq4gkPqkzQI2s+MeVvMN7ga64aRJHl9hm9fceqBN3q5WMivdEQA19v7SAIPbdPRvepRTAyoVkp6RuWB/h6rzlyPBtsuI8X2yMCTw3CVYY0N2qd39CcwP9cEeFDtS07tiKljv4pSbjaqOZzA/yGZw96iwyR50dOj+j7sNoEw7LmkzP/hqMRutUKsbpd5ClWuObcZrXYbzw8Q79UAa+3dKKnUkFvIUKrDFqlDRyYfuiVKcR27PjPKQsjG998XCI4ONPQO1LBQcTXSydzz8oE3QrnljQdjvwbnD8E+gcr+FbkRWGf2v7jChZ0S5xPjwQcjDgEeusPoR1bXp/YeVeqBYPa5dM6vdFQg+BzJYgPOTHHA9/YYzjMI2sYtGuUD9V9wyL/QGvhwbctz+rT+LHff/TULhgZ1XxM8Dc0/A8MFBpcS9ug5rtWLFykV/7YvL7h+YDmuQqRHQd6Rqzdr2hBu1r51Cy/4bC/5sEKfuxHqivun6KF605lVun6xsMMtKoW/NCNA+cCVvBULZicyKySAg7MGPRKFjzXcXr1W+5ThqdkwehA5tOawcEJhN76QeACd9BUPrgD+YJWUfO7mX6Ds4NQ771GUWwr07dPpcI1xi8fzPQbjBiEukclRJatSqs9uijB3AzmA5HeaNB/qFWaEEZOGLp95nk/tQlmJzOfEvkNSwBCvVqFMIrCXkmPPQLwlCqYnsZ8Qvs3NP4PDRQQrNj1HFsPkrt7qYIZhfPJWX7D8watVYXAsgKIxE+VwtCTm4/O+xuOS2hWJgRWHLl9Lgl5UGmCbvVz817uNx52YUCQYEV27PapxT8VCVeY3nxCDTgwzdCsSbA8VymLeWoSRjbC+aQ+cHCqofeipDh2QlvTPfh3o1QwJXw+MflvcDYR6a1CWFYUhHou4HoQtYJh0fOxeYCDsYpIs2zBsm1lE8lDyRa0q5jVmr3R0IVIv0ghsH0H2EL/W1QfUqWgVZ/c5hg7yr5huUCkt+Tghk4U9xoNs9909NQpmBjTfFrnNzQBiPSOVvCcIHRV1a+nTsHofOYPTOsbnhPon9Qc2HHsu8+JC9cdznxirt9w+F9zVSGK3dDTNG3tQbQLRkTOKpHfePiCXumCE3hxFPXwZk/pgvGJzGe1gQMzCr0qBjtwA0sZV58yhgEnM58SDQ5OJPROY44A1tl6brm8A82CjjHMR4f7DU4MYr3FgtCOXE3T/R5Ds2BI3Px+4N9g7CDWWx3w4jhwNN2Gei/6BVODmY9L9UZDA2K9GobYizyrTwf5U8NgeFxzV+03LN6P9d5zhIr5wOrRt7EH95+iBdMDmg/0gENj/lhv8QBQWxBYei6lvHuNgubZzKohfsMTA72aBM+NI8vWwwweUJTQX818bsLfcLxA84VHsWU58VOLMI4ZzUcUfuOhDXqrB7YTxH7/qWxPWcJVJjaflgIOTCr0FhH8OHKU1aynRGE845uP6gMH5xZ6L0Gyg8jqlVS5Z2WCGZ1z814QOAo6YVsGBim4URiHeupUj6pM0CN0PjMG0CjF+D//H0jWvvo='))
assert hashlib.sha256(embedded_payload).hexdigest() == '5fc4ea0c61bd3e3e475cbbff7decf8c761f7cdb89cc1f68aad1250006ba25d58'
embedded_frame = pd.DataFrame(json.loads(embedded_payload))
PHASE1_INPUT_PATH.parent.mkdir(parents=True, exist_ok=True)
if PHASE1_INPUT_PATH.exists():
    existing_frame = pd.read_csv(PHASE1_INPUT_PATH)
    pd.testing.assert_frame_equal(
        existing_frame[embedded_frame.columns].reset_index(drop=True),
        embedded_frame, check_dtype=False, check_exact=False, rtol=1e-12, atol=1e-12,
    )
else:
    embedded_frame.to_csv(PHASE1_INPUT_PATH, index=False)
print("Verified embedded Mixed Emotion input: 300 rows, 86 routed.")

if not PHASE1_INPUT_PATH.exists():
    raise FileNotFoundError(f"Final Phase 1 export not found: {PHASE1_INPUT_PATH}")


def normalize_bool(value):
    if isinstance(value, bool):
        return value
    text = str(value).strip().lower()
    if text in {"true", "1", "yes", "y"}:
        return True
    if text in {"false", "0", "no", "n"}:
        return False
    raise ValueError(f"Unrecognized Boolean value: {value!r}")


def normalize_label(value):
    text = str(value).strip().lower()
    for label in LABELS:
        if text == label.lower():
            return label
    return np.nan


def minimally_sanitize_original_text(title, selftext):
    title = "" if pd.isna(title) else str(title).strip()
    body = "" if pd.isna(selftext) else str(selftext).strip()
    if body.lower() in {"[deleted]", "[removed]", "nan", "none"}:
        body = ""
    combined = title if not body else f"{title}\n\n{body}"
    combined = re.sub(r"https?://\S+|www\.\S+", " [URL] ", combined, flags=re.I)
    combined = re.sub(r"(?<!\w)(?:/u/|u/)[A-Za-z0-9_-]+", "[USER]", combined)
    combined = combined.replace("\r\n", "\n").replace("\r", "\n")
    combined = re.sub(r"[ \t]+", " ", combined)
    return re.sub(r"\n{3,}", "\n\n", combined).strip()


def limit_reasoning_text(value):
    value = "" if pd.isna(value) else str(value)
    if len(value) <= MAX_REASONING_CHARACTERS:
        return value, False
    marker = "\n\n[Middle omitted to fit model context; beginning and ending preserved.]\n\n"
    available = MAX_REASONING_CHARACTERS - len(marker)
    head_characters = min(REASONING_HEAD_CHARACTERS, available)
    tail_characters = min(REASONING_TAIL_CHARACTERS, available - head_characters)
    return (
        value[:head_characters]
        + marker
        + value[-tail_characters:],
        True,
    )


def normalize_phase1_export(path):
    frame = pd.read_csv(path)
    aliases = {
        "target_label": ["target_label", "label_str", "true_label", "reference_label"],
        "phase1_label": ["phase1_label", "prediction", "predicted_label"],
        "phase1_confidence": ["phase1_confidence", "calibrated_confidence", "confidence"],
        "phase1_routed": ["phase1_routed", "routed", "route_indicator"],
        "phase2_original_text": ["phase2_original_text", "original_text", "reasoning_text"],
    }
    for canonical, candidates in aliases.items():
        if canonical in frame.columns:
            continue
        found = next((candidate for candidate in candidates if candidate in frame.columns), None)
        if found:
            frame = frame.rename(columns={found: canonical})

    if "phase2_original_text" not in frame.columns:
        if {"title", "selftext"}.issubset(frame.columns):
            frame["phase2_original_text"] = [
                minimally_sanitize_original_text(title, body)
                for title, body in zip(frame["title"], frame["selftext"])
            ]
        else:
            raise ValueError(
                "Phase 1 export needs phase2_original_text, or both title and selftext."
            )

    required = {
        "example_id", "target_label", "phase1_label", "phase1_confidence",
        "phase1_routed", "phase2_original_text",
    }
    missing = required - set(frame.columns)
    if missing:
        raise ValueError(f"Final Phase 1 export is missing columns: {sorted(missing)}")

    frame["example_id"] = frame["example_id"].astype(str)
    if frame["example_id"].duplicated().any():
        duplicates = frame.loc[frame["example_id"].duplicated(), "example_id"].head().tolist()
        raise ValueError(f"Duplicate example_id values found: {duplicates}")

    frame["target_label"] = frame["target_label"].map(normalize_label)
    frame["phase1_label"] = frame["phase1_label"].map(normalize_label)
    frame["phase1_routed"] = frame["phase1_routed"].map(normalize_bool)
    frame["phase1_confidence"] = pd.to_numeric(frame["phase1_confidence"], errors="raise")
    if frame[["target_label", "phase1_label"]].isna().any().any():
        raise ValueError("target_label and phase1_label must use Depression, Neutral, or Happy.")
    if not frame["phase1_confidence"].between(0, 1).all():
        raise ValueError("phase1_confidence must be between 0 and 1.")

    if "phase2_input_was_truncated" in frame.columns:
        previously_truncated = (
            frame["phase2_input_was_truncated"].fillna(False).map(normalize_bool)
        )
    else:
        previously_truncated = pd.Series(False, index=frame.index)
    limited = frame["phase2_original_text"].map(limit_reasoning_text)
    frame["phase2_original_text"] = limited.map(lambda pair: pair[0])
    frame["phase2_input_was_truncated"] = (
        previously_truncated | limited.map(lambda pair: pair[1])
    )

    routed = frame[frame["phase1_routed"]].copy()
    if routed.empty:
        raise ValueError("No routed rows were found in the final Phase 1 export.")
    if routed["phase2_original_text"].str.strip().eq("").any():
        raise ValueError("At least one routed row has empty Phase 2 original text.")
    if EXPECTED_TOTAL_ROWS is not None and len(frame) != EXPECTED_TOTAL_ROWS:
        raise ValueError(f"Expected {EXPECTED_TOTAL_ROWS} total rows, found {len(frame)}.")
    if EXPECTED_ROUTED_ROWS is not None and len(routed) != EXPECTED_ROUTED_ROWS:
        raise ValueError(f"Expected {EXPECTED_ROUTED_ROWS} routed rows, found {len(routed)}.")

    if MAX_ROWS is not None:
        routed = routed.head(min(MAX_ROWS, len(routed))).copy()
    return frame.reset_index(drop=True), routed.reset_index(drop=True)


phase1_df, routed_df = normalize_phase1_export(PHASE1_INPUT_PATH)
routed_id_hash = hashlib.sha256(
    "\n".join(sorted(routed_df["example_id"])).encode("utf-8")
).hexdigest()
print("Phase 1 rows:", len(phase1_df))
print("Routed rows selected for this run:", len(routed_df))
print("Routed example_id SHA-256:", routed_id_hash)
display(routed_df[["example_id", "target_label", "phase1_label", "phase1_confidence"]].head())


## Model, generation, persistence, and evaluation helpers


In [ ]:
def stable_seed(example_id, method, stage):
    key = f"{BASE_SEED}:{example_id}:{method}:{stage}".encode("utf-8")
    return int.from_bytes(hashlib.sha256(key).digest()[:4], "big")


def set_generation_seed(seed):
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def load_chat_model(model_name):
    tokenizer = AutoTokenizer.from_pretrained(
        model_name, trust_remote_code=True, token=os.environ.get("HF_TOKEN")
    )
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token
    tokenizer.padding_side = "right"
    quantization_config = None
    if LOAD_IN_4BIT:
        quantization_config = BitsAndBytesConfig(
            load_in_4bit=True,
            bnb_4bit_quant_type="nf4",
            bnb_4bit_compute_dtype=torch.float16,
            bnb_4bit_use_double_quant=True,
        )
    model = AutoModelForCausalLM.from_pretrained(
        model_name,
        torch_dtype=torch.float16,
        device_map="auto",
        quantization_config=quantization_config,
        trust_remote_code=True,
        token=os.environ.get("HF_TOKEN"),
    )
    model.eval()
    return tokenizer, model


def format_messages_without_chat_template(messages):
    system_parts = [m["content"] for m in messages if m.get("role") == "system"]
    dialogue = [m for m in messages if m.get("role") != "system"]
    system_text = "\n".join(system_parts).strip()
    prompt, pending_user, first_turn = "", None, True
    for message in dialogue:
        role = message.get("role")
        content = str(message.get("content", "")).strip()
        if role == "user":
            pending_user = content
        elif role == "assistant" and pending_user is not None:
            if first_turn and system_text:
                prompt += f"<s>[INST] <<SYS>>\n{system_text}\n<</SYS>>\n\n{pending_user} [/INST] {content} </s>"
            else:
                prompt += f"<s>[INST] {pending_user} [/INST] {content} </s>"
            pending_user, first_turn = None, False
    if pending_user is not None:
        if first_turn and system_text:
            prompt += f"<s>[INST] <<SYS>>\n{system_text}\n<</SYS>>\n\n{pending_user} [/INST]"
        else:
            prompt += f"<s>[INST] {pending_user} [/INST]"
    return prompt


def build_chat_inputs(tokenizer, model, messages):
    if getattr(tokenizer, "chat_template", None):
        prompt = tokenizer.apply_chat_template(
            messages, tokenize=False, add_generation_prompt=True
        )
    else:
        prompt = format_messages_without_chat_template(messages)
    encoded = tokenizer(prompt, return_tensors="pt")
    return {key: value.to(model.device) for key, value in encoded.items()}


def chat_generate(
    tokenizer, model, messages, *, max_new_tokens, seed,
    do_sample=False, temperature=None, top_p=None,
):
    set_generation_seed(seed)
    model_inputs = build_chat_inputs(tokenizer, model, messages)
    input_ids = model_inputs["input_ids"]
    terminators = [tokenizer.eos_token_id]
    if "<|eot_id|>" in tokenizer.get_vocab():
        eot_id = tokenizer.convert_tokens_to_ids("<|eot_id|>")
        if isinstance(eot_id, int) and eot_id >= 0 and eot_id not in terminators:
            terminators.append(eot_id)
    kwargs = dict(
        **model_inputs,
        max_new_tokens=max_new_tokens,
        eos_token_id=terminators,
        pad_token_id=tokenizer.eos_token_id,
        do_sample=do_sample,
    )
    if do_sample and temperature is not None:
        kwargs["temperature"] = temperature
    if do_sample and top_p is not None:
        kwargs["top_p"] = top_p
    if torch.cuda.is_available():
        torch.cuda.synchronize()
    started = time.perf_counter()
    with torch.inference_mode():
        outputs = model.generate(**kwargs)
    if torch.cuda.is_available():
        torch.cuda.synchronize()
    elapsed = time.perf_counter() - started
    generated = outputs[0][input_ids.shape[-1]:]
    return {
        "text": tokenizer.decode(generated, skip_special_tokens=True).strip(),
        "input_tokens": int(input_ids.shape[-1]),
        "generated_tokens": int(generated.shape[-1]),
        "elapsed_seconds": float(elapsed),
        "hit_token_limit": bool(generated.shape[-1] >= max_new_tokens),
    }


def parse_final_label(output):
    text = str(output)
    match = re.search(
        r"Final\s+label\s*\*{0,2}\s*:\s*\*{0,2}\s*"
        r"(Depression|Neutral|Happy)\b",
        text,
        flags=re.I,
    )
    if match:
        return normalize_label(match.group(1))

    # Some models return a clear terminal label without the requested prefix.
    # Recover it only when the final-stage answer contains one unique allowed
    # label; refusals and outputs mentioning multiple labels remain unparsed.
    labels = {
        normalize_label(label)
        for label in re.findall(r"\b(Depression|Neutral|Happy)\b", text, flags=re.I)
    }
    return next(iter(labels)) if len(labels) == 1 else np.nan


def load_results(path):
    if path.exists() and path.stat().st_size > 0:
        frame = pd.read_csv(path)
        return frame.drop_duplicates(subset=["example_id"], keep="last")
    return pd.DataFrame()


def save_result(path, row):
    path.parent.mkdir(parents=True, exist_ok=True)
    existing = load_results(path)
    updated = pd.concat([existing, pd.DataFrame([row])], ignore_index=True)
    updated = updated.drop_duplicates(subset=["example_id"], keep="last")
    temporary = path.with_suffix(".tmp.csv")
    updated.to_csv(temporary, index=False)
    os.replace(temporary, path)


def clear_model(tokenizer, model):
    del tokenizer, model
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()


def summarize_method(full_phase1, routed_results, prediction_col, method):
    processed_ids = set(routed_results["example_id"].astype(str))
    evaluation_base = full_phase1
    scope = "full_end_to_end"
    if MAX_ROWS is not None:
        evaluation_base = full_phase1[full_phase1["example_id"].isin(processed_ids)].copy()
        scope = "smoke_routed_subset"
    merged = evaluation_base.merge(
        routed_results[["example_id", prediction_col]],
        on="example_id", how="left", validate="one_to_one",
    )
    merged["final_label"] = merged["phase1_label"]
    routed_mask = merged["phase1_routed"]
    parsed_routed_mask = routed_mask & merged[prediction_col].isin(LABELS)
    merged.loc[parsed_routed_mask, "final_label"] = merged.loc[
        parsed_routed_mask, prediction_col
    ]
    merged["phase2_parse_success"] = ~routed_mask | parsed_routed_mask
    merged["phase2_fallback_to_phase1"] = routed_mask & ~parsed_routed_mask
    merged["phase1_correct"] = merged["phase1_label"].eq(merged["target_label"])
    merged["final_correct"] = merged["final_label"].eq(merged["target_label"])
    corrected = int((routed_mask & ~merged["phase1_correct"] & merged["final_correct"]).sum())
    introduced = int((routed_mask & merged["phase1_correct"] & ~merged["final_correct"]).sum())
    parsed = routed_results[prediction_col].isin(LABELS)
    summary = {
        "method": method,
        "scope": scope,
        "total_rows": len(merged),
        "routed_rows": int(routed_mask.sum()),
        "parsed_routed_rows": int(parsed.sum()),
        "parse_failures": int((~parsed).sum()),
        "phase1_accuracy": float(merged["phase1_correct"].mean()),
        "end_to_end_accuracy": float(merged["final_correct"].mean()),
        "end_to_end_macro_f1": float(
            f1_score(merged["target_label"], merged["final_label"], labels=LABELS, average="macro")
        ),
        "corrected": corrected,
        "introduced": introduced,
        "net_corrections": corrected - introduced,
    }
    return merged, summary


## Llama 3-matched Llama 2 CoT prompt and runner


In [ ]:
LLAMA2_MODEL_NAME = "NousResearch/Llama-2-7b-chat-hf"
LLAMA2_OUTPUT_DIR = DRIVE_OUTPUT_ROOT / "llama2_cot_matched_llama3"
LLAMA2_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
LLAMA2_RESULTS_PATH = LLAMA2_OUTPUT_DIR / "llama2_cot_matched_results.csv"
LLAMA2_PROMPT_VERSION = "final-llama2-cot-v3-matched-llama3"
MAX_NEW_TOKENS_COT = 512

LLAMA2_CLASSIFICATION_POLICY = """Classification Policy:
- Depression: unresolved sadness, hopelessness, emotional distress, emotional exhaustion, withdrawal, self-devaluation, or a clearly negative overall trajectory is dominant.
- Neutral: the text is mainly factual, routine, balanced, informational, or emotionally mild, without a dominant positive or distress-related state.
- Happy: happiness, relief, gratitude, accomplishment, fulfillment, or a clearly positive resolution is dominant.

Assess the dominant emotional meaning of the full text. For mixed or shifting emotions, consider the overall trajectory and final takeaway. Do not decide from isolated words, and do not make clinical diagnoses or treatment recommendations."""

LLAMA2_COT_REQUESTS = [
    "You are an expert annotator for research-oriented, non-clinical emotion classification. Assist in analyzing emotions in text data. Do not make a clinical diagnosis, infer a medical condition, or provide treatment advice.",
    "I will first provide a piece of text. Independently assess its emotional content before comparing it with a Phase 1 AI-generated label. The only permitted labels are Depression, Neutral, and Happy.",
    "Independently analyze the text. State the dominant emotion and textual evidence before considering the Phase 1 label.\n\n{policy}\n\nUse only Depression, Neutral, or Happy for your provisional label.",
    """The Phase 1 classifier predicted: {phase1_label}

Compare that prediction with your independent assessment. Confirm it only when it is supported by the dominant emotional meaning of the full text. Otherwise, explain the correction using textual evidence only.""",
    "Provide the final Phase 2 decision. Use only one exact label: Depression, Neutral, or Happy. Do not use synonyms or additional labels. End the response with exactly one line: Final label: [label]",
]


def run_llama2_cot_one(tokenizer, model, row):
    example_id = row["example_id"]
    messages = [
        {"role": "system", "content": LLAMA2_COT_REQUESTS[0]},
        {"role": "user", "content": LLAMA2_COT_REQUESTS[1]},
    ]
    stages = []

    ack = chat_generate(
        tokenizer, model, messages, max_new_tokens=MAX_NEW_TOKENS_COT,
        seed=stable_seed(example_id, "llama2_cot", "ack"),
        do_sample=False,
    )
    stages.append(ack)
    messages.append({"role": "assistant", "content": ack["text"]})
    messages.append({"role": "user", "content": f"Text:\n{row['phase2_original_text']}"})

    text_response = chat_generate(
        tokenizer, model, messages, max_new_tokens=MAX_NEW_TOKENS_COT,
        seed=stable_seed(example_id, "llama2_cot", "text_response"),
        do_sample=False,
    )
    stages.append(text_response)
    messages.append({"role": "assistant", "content": text_response["text"]})
    messages.append({
        "role": "user",
        "content": LLAMA2_COT_REQUESTS[2].format(
            policy=LLAMA2_CLASSIFICATION_POLICY
        ),
    })

    independent = chat_generate(
        tokenizer, model, messages, max_new_tokens=MAX_NEW_TOKENS_COT,
        seed=stable_seed(example_id, "llama2_cot", "independent"),
        do_sample=False,
    )
    stages.append(independent)
    messages.append({"role": "assistant", "content": independent["text"]})
    messages.append({"role": "user", "content": LLAMA2_COT_REQUESTS[3].format(phase1_label=row["phase1_label"])})

    comparison = chat_generate(
        tokenizer, model, messages, max_new_tokens=MAX_NEW_TOKENS_COT,
        seed=stable_seed(example_id, "llama2_cot", "comparison"),
        do_sample=False,
    )
    stages.append(comparison)
    messages.append({"role": "assistant", "content": comparison["text"]})
    messages.append({"role": "user", "content": LLAMA2_COT_REQUESTS[4]})

    final = chat_generate(
        tokenizer, model, messages, max_new_tokens=MAX_NEW_TOKENS_COT,
        seed=stable_seed(example_id, "llama2_cot", "final"),
        do_sample=False,
    )
    stages.append(final)
    return {
        "llama2_ack": ack["text"],
        "llama2_text_response": text_response["text"],
        "llama2_independent": independent["text"],
        "llama2_comparison": comparison["text"],
        "llama2_final_answer": final["text"],
        "llama2_final_label": parse_final_label(final["text"]),
        "input_tokens_total": sum(item["input_tokens"] for item in stages),
        "generated_tokens_total": sum(item["generated_tokens"] for item in stages),
        "generation_seconds_total": sum(item["elapsed_seconds"] for item in stages),
        "any_stage_hit_token_limit": any(item["hit_token_limit"] for item in stages),
    }


## Run or resume Llama 2 CoT


In [ ]:
results = load_results(LLAMA2_RESULTS_PATH)
if not RESUME_FROM_EXISTING and LLAMA2_RESULTS_PATH.exists():
    LLAMA2_RESULTS_PATH.unlink()
    results = pd.DataFrame()
completed = set(results.get("example_id", pd.Series(dtype=str)).astype(str))
pending = routed_df[~routed_df["example_id"].isin(completed)].copy()
print(f"Llama 2 CoT: {len(completed)} completed, {len(pending)} pending")

if not pending.empty:
    tokenizer, model = load_chat_model(LLAMA2_MODEL_NAME)
    model_revision = getattr(model.config, "_commit_hash", None)
    for _, row in tqdm(pending.iterrows(), total=len(pending), desc="Llama 2 CoT"):
        generated = run_llama2_cot_one(tokenizer, model, row)
        result = {
            "example_id": row["example_id"],
            "target_label": row["target_label"],
            "phase1_label": row["phase1_label"],
            "phase1_confidence": row["phase1_confidence"],
            "phase2_original_text": row["phase2_original_text"],
            "phase2_input_was_truncated": row["phase2_input_was_truncated"],
            "model_name": LLAMA2_MODEL_NAME,
            "model_revision": model_revision,
            "prompt_version": LLAMA2_PROMPT_VERSION,
            "routed_id_hash": routed_id_hash,
            **generated,
        }
        save_result(LLAMA2_RESULTS_PATH, result)
    clear_model(tokenizer, model)

results = load_results(LLAMA2_RESULTS_PATH)
if len(results) != len(routed_df) or set(results["example_id"].astype(str)) != set(routed_df["example_id"]):
    raise ValueError("Llama 2 result IDs do not exactly match the routed input IDs.")
end_to_end, summary = summarize_method(
    phase1_df, results, "llama2_final_label", "Llama 2 CoT matched"
)
end_to_end.to_csv(LLAMA2_OUTPUT_DIR / "llama2_cot_matched_end_to_end_predictions.csv", index=False)
pd.DataFrame([summary]).to_csv(LLAMA2_OUTPUT_DIR / "llama2_cot_matched_summary.csv", index=False)
display(pd.DataFrame([summary]))
display(results[["example_id", "target_label", "phase1_label", "llama2_final_label"]].head())
print("Saved:", LLAMA2_OUTPUT_DIR)
